In [63]:
from modules.strategies import Base_RSI
from backtesting_engine.backtest_engine import BacktestEngine
from modules.data import generate_historical_data
from datetime import datetime, timedelta

import cProfile
import pstats

strategy_configs = {
    'RSI': {
        'stream_key': 'alpaca',
        'tickers': ['AAPL','AMD','CSCO','GOOG','INTC','JNPR','META','MSFT','NFLX','NVDA','TSLA'],  # List of tickers to trade
        'hist_period': 14,  # Historical data required per new price entry
        'overbought_th': {'entry': 55, 'exit':50},  # Overbought threshold for RSI
        'oversold_th': {'entry': 45, 'exit':50},  # Oversold threshold for RSI
        'plot':False,
    },
}

strategy_name = 'RSI'

In [75]:
tickers =  strategy_configs[strategy_name]['tickers'][0]
start_time = datetime.now() - timedelta(days=1)  
end_time = datetime.now()
historical_data = generate_historical_data(tickers, start_time, end_time)
historical_data.sort_values(by='timestamp', inplace=True)
historical_data = historical_data.iloc[:25]
historical_data.head()

,timestamp,ticker,open,high,low,close
0,2025-03-28 01:17:57.924234,AAPL,316.61,319.18,308.65,312.50
1,2025-03-28 01:18:57.924234,AAPL,312.79,314.27,309.77,310.05
2,2025-03-28 01:19:57.924234,AAPL,303.45,312.25,301.17,308.20
3,2025-03-28 01:20:57.924234,AAPL,308.89,316.88,308.38,313.01
4,2025-03-28 01:21:57.924234,AAPL,302.11,307.87,300.77,306.43


In [76]:
import pandas as pd
import numpy as np


def rsi2(prices, period=14):
    """
    Computes RSI in a length-invariant way using Wilder's smoothing.
    This ensures consistency for both small and large datasets.

    Args:
        prices (np.ndarray): Array of closing prices.
        period (int): RSI period (default: 14).

    Returns:
        np.ndarray: RSI values with proper smoothing applied.
    """
    prices = np.asarray(prices, dtype=np.float64)
    
    if len(prices) < period:
        return np.full_like(prices, np.nan, dtype=np.float64)  # Not enough data

    delta = np.diff(prices, prepend=prices[0])
    gains = np.where(delta > 0, delta, 0)
    losses = np.where(delta < 0, -delta, 0)

    # Compute the first average gain/loss over the first `period` values
    avg_gain = np.mean(gains[:period])
    avg_loss = np.mean(losses[:period])

    # Initialize RSI array
    rsi_values = np.full_like(prices, np.nan, dtype=np.float64)

    # Apply Wilder’s smoothing for the rest of the dataset
    smoothing_factor = (period - 1) / period

    for i in range(period, len(prices)):
        avg_gain = (avg_gain * smoothing_factor) + (gains[i] / period)
        avg_loss = (avg_loss * smoothing_factor) + (losses[i] / period)

        rs = avg_gain / (avg_loss if avg_loss != 0 else np.inf)  # Avoid division by zero
        rsi_values[i] = 100 - (100 / (1 + rs))

    return rsi_values

In [77]:
period = 14
def rsi (close_prices, period=14):

    delta = np.diff(close_prices)
    
    gains = np.where(delta > 0, delta, 0)
    losses = np.where(delta < 0, -delta, 0)
    
    avg_gain = np.convolve(gains, np.ones(period), mode='valid')/period
    avg_loss = np.convolve(losses, np.ones(period), mode='valid')/period
    
    avg_gain = np.concatenate((np.full(period, np.nan), avg_gain))
    avg_loss = np.concatenate((np.full(period, np.nan), avg_loss))
    
    
    avg_loss[avg_loss == 0] = np.inf
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    
    return rsi


In [78]:
from collections import deque

historical_data.sort_values(by='timestamp', inplace=True)
rsi_vals = rsi(historical_data.close.to_numpy(), period)
historical_data['values_df'] = rsi_vals

window = deque(maxlen=period+1)
values = []
for row in historical_data.itertuples(index=False):
    ticker = row.ticker
    window.append(row.close)
    if len(window) < period+1:
        values.append(np.nan)
        continue        
    rsi_val = rsi(np.array(window), period)[-1]
    values.append(rsi_val)   
historical_data['values_window'] = values

In [79]:
#historical_data['manual'] = df_list['rsi']
historical_data.head(25)

,timestamp,ticker,open,high,low,close,values_df,values_window
0,2025-03-28 01:17:57.924234,AAPL,316.61,319.18,308.65,312.50,NaN,NaN
1,2025-03-28 01:18:57.924234,AAPL,312.79,314.27,309.77,310.05,NaN,NaN
2,2025-03-28 01:19:57.924234,AAPL,303.45,312.25,301.17,308.20,NaN,NaN
3,2025-03-28 01:20:57.924234,AAPL,308.89,316.88,308.38,313.01,NaN,NaN
4,2025-03-28 01:21:57.924234,AAPL,302.11,307.87,300.77,306.43,NaN,NaN
5,2025-03-28 01:22:57.924234,AAPL,302.59,307.19,300.69,301.68,NaN,NaN
6,2025-03-28 01:23:57.924234,AAPL,306.46,312.50,305.07,308.71,NaN,NaN
7,2025-03-28 01:24:57.924234,AAPL,310.87,313.64,309.00,310.29,NaN,NaN
8,2025-03-28 01:25:57.924234,AAPL,299.18,306.96,296.60,303.04,NaN,NaN
9,2025-03-28 01:26:57.924234,AAPL,299.87,301.63,295.27,300.22,NaN,NaN


In [ ]:
### delta = np.diff(close_prices, prepend=close_prices[0])  # Ensure same length
gains = np.where(delta > 0, delta, 0)
losses = np.where(delta < 0, -delta, 0)

# Initial average gain/loss using simple mean
avg_gain = np.convolve(gains, np.ones(period) / period, mode='valid')
avg_loss = np.convolve(losses, np.ones(period) / period, mode='valid')

# Expand to match the original array size with NaN for the first period-1 values
avg_gain = np.concatenate((np.full(period-1, np.nan), avg_gain))
avg_loss = np.concatenate((np.full(period-1, np.nan), avg_loss))

# Apply Wilder's smoothing for each value after the initial period
for i in range(period, len(close_prices)):
    avg_gain[i] = (avg_gain[i - 1] * (period - 1) + gains[i]) / period
    avg_loss[i] = (avg_loss[i - 1] * (period - 1) + losses[i]) / period

# Prevent division by zero
avg_loss[avg_loss == 0] = 1e-10  # Prevent division by zero
rs = avg_gain / avg_loss
rsi = 100 - (100 / (1 + rs))